In [339]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [ ]:
# ============================================================  
# CHANGE THESE TWO DATES EACH TIME YOU RUN
# ============================================================  
semi_monthly_date = '2026-08-15'  
biweekly_date     = '2026-08-22'  
# ============================================================

# read excel sheet with a password
def read_protected_excel(file_path, password, sheet_names, skiprows_list):  
    """Read multiple sheets from a password-protected Excel file."""  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        office_file.load_key(password=password)  
        decrypted = io.BytesIO()  
        office_file.decrypt(decrypted)

    dfs = []  
    for i, sheet in enumerate(sheet_names):  
        decrypted.seek(0)  
        df = pd.read_excel(decrypted, sheet_name=sheet, skiprows=skiprows_list[i])  
        dfs.append(df)

    return dfs
  
# set the file path for manpower files
base_path = 'L:/2026 Pilar Plant Files/2026 Manpower Detail/'  
password = 'bamhorse2'  
# set the tabs that are needed
sheet_names = ['YTD FTEs' , 'MTD LDR', 'PPE FTEs']  
# add the number of rows to skip at the top of each sheet (to ignore headers)
skiprows_list = [7, 5, 8]

# find all files matching each date
semi_monthly_files = glob.glob(os.path.join(base_path, f'*{semi_monthly_date}*.xlsm'))  
biweekly_files = glob.glob(os.path.join(base_path, f'*{biweekly_date}*.xlsm'))

print(f"Semi-Monthly files ({semi_monthly_date}):")  
for f in semi_monthly_files:  
    print(f"  {os.path.basename(f)}")

print(f"\nBiweekly files ({biweekly_date}):")  
for f in biweekly_files:  
    print(f"  {os.path.basename(f)}")

# combine into one list with pay type labels  
files = [(f, 'Semi-Monthly', semi_monthly_date) for f in semi_monthly_files] + [(f, 'Biweekly', biweekly_date) for f in biweekly_files]

all_ytd_ftes = []  
all_mtd_ldr = []  
all_ppe_ftes = []

# for loop!
for file_path, pay_type, date in files:  
    # extract site name from filename (everything before the date)  
    filename = os.path.basename(file_path)  
    site = filename.split(' ')[0].strip()  
    
    print(f"\nReading {pay_type} | Site: {site} | Date: {date}")

    try:  
        dfs = read_protected_excel(file_path, password, sheet_names, skiprows_list)
        for df in dfs:
            # adding extra columns  
            df['Pay_Type'] = pay_type  
            df['Report_Date'] = pd.to_datetime(date)  
            df['BU'] = site

        # appending data to dataframe
        all_ytd_ftes.append(dfs[0])  
        all_mtd_ldr.append(dfs[1])  
        all_ppe_ftes.append(dfs[2])

    except Exception as e:  
        print(f"  ERROR: {e}")

# combines lists of dataframes into one dataframe for each tab
ytd_ftes_combined = pd.concat(all_ytd_ftes, ignore_index=True)  
mtd_ldr_combined = pd.concat(all_mtd_ldr, ignore_index=True)  
ppe_ftes_combined = pd.concat(all_ppe_ftes, ignore_index=True)

# renames columns to preferred format
ytd_ftes_combined.columns = ytd_ftes_combined.columns.str.strip().str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.replace(r'_+', '_', regex=True).str.strip('_').str.lower()  
mtd_ldr_combined.columns = mtd_ldr_combined.columns.str.strip().str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.replace(r'_+', '_', regex=True).str.strip('_').str.lower()  
ppe_ftes_combined.columns = ppe_ftes_combined.columns.str.strip().str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.replace(r'_+', '_', regex=True).str.strip('_').str.lower()  

print(f"\n--- Summary ---")  
print(f"YTD FTEs: {ytd_ftes_combined.shape}")  
print(f"MTD LDR:  {mtd_ldr_combined.shape}")  
print(f"PPE FTEs: {ppe_ftes_combined.shape}")  
print(f"Sites found: {ytd_ftes_combined['bu'].unique()}")  

Semi-Monthly files (2026-08-15):
  GVCCC SALAERN GVCCC_EXEC 2026-08-15.xlsm
  LENOX SALAERN LENOX_EXEC 2026-08-15.xlsm
  MEETH SALAERN MEETH_EXEC 2026-08-15.xlsm

Biweekly files (2026-08-22):
  GVCCC SALAERN GVCCC_EXEC 2026-08-22.xlsm
  LENOX SALAERN LENOX_EXEC 2026-08-22.xlsm
  MEETH SALAERN MEETH_EXEC 2026-08-22.xlsm

Reading Semi-Monthly | Site: GVCCC | Date: 2026-08-15

Reading Semi-Monthly | Site: LENOX | Date: 2026-08-15

Reading Semi-Monthly | Site: MEETH | Date: 2026-08-15

Reading Biweekly | Site: GVCCC | Date: 2026-08-22

Reading Biweekly | Site: LENOX | Date: 2026-08-22

Reading Biweekly | Site: MEETH | Date: 2026-08-22

--- Summary ---
YTD FTEs: (3795, 20)
MTD LDR:  (17357, 35)
PPE FTEs: (2344, 20)
Sites found: <StringArray>
['GVCCC', 'LENOX', 'MEETH']
Length: 3, dtype: str


In [341]:
# dates to datetime objects
semi_monthly_dt = pd.to_datetime(semi_monthly_date)  
biweekly_dt = pd.to_datetime(biweekly_date)
# most recent report date
most_recent_report_date = max(semi_monthly_dt, biweekly_dt)  

# column to datetime
mtd_ldr_combined['pay_end_date'] = pd.to_datetime(mtd_ldr_combined['pay_end_date'])

# condition for ldr month consistency
# if in the same month
if semi_monthly_dt.month == biweekly_dt.month:  
    # create a cutoff date 14 days prior to most recent date  
    cutoff_date = most_recent_report_date - pd.Timedelta(days=14)

    # print notification
    print(f"Same month. Using only most recent file date: {most_recent_report_date}")  
    print(f"Filtering Pay End Date between {cutoff_date} and {most_recent_report_date}")

    # only use most recent file and filter pay end dates to between most recent ppe date and prior ppe date
    mtd_ldr_combined = mtd_ldr_combined[  
        (mtd_ldr_combined['report_date'] == most_recent_report_date) &  
        (mtd_ldr_combined['pay_end_date'] > cutoff_date) &  
        (mtd_ldr_combined['pay_end_date'] <= most_recent_report_date)
    ]

# if the files are in two different months
else:
    # create a cutoff date 14 days prior to most recent date  
    cutoff_date = most_recent_report_date - pd.Timedelta(days=14)

    # print notification
    print(f"Different months. Using both files.")  
    print(f"Filtering Pay End Date between {cutoff_date} and {most_recent_report_date}")

    # use both files and filter pay end dates to between most recent ppe date and prior ppe date
    mtd_ldr_combined = mtd_ldr_combined[  
        (mtd_ldr_combined['pay_end_date'] > cutoff_date) &  
        (mtd_ldr_combined['pay_end_date'] <= most_recent_report_date)  
    ]  

# get a list of report dates included
report_dates = mtd_ldr_combined['report_date'].dt.strftime('%Y-%m-%d').unique().tolist()  
# get a list of pay end dates included
pay_end_dates = sorted(mtd_ldr_combined['pay_end_date'].dt.strftime('%Y-%m-%d').unique().tolist())

# print which report dates and pay end dates were included
print(f"Report date(s): {', '.join(report_dates)}")  
print(f"Pay End Dates:  {', '.join(pay_end_dates)}")  

# add fte and type columns
mtd_ldr_combined['reg_ftes'] = (mtd_ldr_combined['total_hours'] - mtd_ldr_combined['ot_hours'])/75
mtd_ldr_combined['ot_ftes'] = mtd_ldr_combined['ot_hours']/75
mtd_ldr_combined['total_ftes'] = mtd_ldr_combined['total_hours']/75
mtd_ldr_combined['type'] = 'Actual'

# filter to selected columns
mtd_ldr_f = mtd_ldr_combined[['type', 'report_date', 'bu', 'gl_deptid', 'description', 'jobcode', 'descr', 'reg_ftes', 'ot_ftes', 'total_ftes']]
# drop nulls then group by all and sum fte columns
mtd_ldr_c = mtd_ldr_f.dropna(subset=['gl_deptid']).groupby(['type', 'report_date', 'bu', 'gl_deptid', 'description', 'jobcode', 'descr'])[['reg_ftes', 'ot_ftes', 'total_ftes']].sum().reset_index()  
# split to flexstaff job codes and dialysis departments
mtd_ldr_flex = mtd_ldr_c[mtd_ldr_c['descr'].str.contains('flex', case=False, na=False)].reset_index(drop=True)
mtd_ldr_dial = mtd_ldr_c[mtd_ldr_c['description'].str.contains('Dialysis', case=False, na=False)].reset_index(drop=True)

# check totals
#mtd_ldr_dial.groupby('bu')[['reg_ftes', 'ot_ftes', 'total_ftes']].sum()

Same month. Using only most recent file date: 2026-08-22 00:00:00
Filtering Pay End Date between 2026-08-08 00:00:00 and 2026-08-22 00:00:00
Report date(s): 2026-08-22
Pay End Dates:  2026-08-14, 2026-08-15, 2026-08-21, 2026-08-22


In [342]:
# function to clean ftes tabs
def clean_ftes(sheet):
    # remove subtotal rowa
    sheet_c = sheet[~sheet['deptid'].str.contains('total', case=False, na=False)]

    # forward fill department ids and names
    sheet_c['deptid'] = sheet_c['deptid'].ffill()
    sheet_c['department_description'] = sheet_c['department_description'].ffill()

    # remove department ids that are blank
    sheet_n = sheet_c[sheet_c['deptid'] != '(blank)']
    # set type to integer
    sheet_n[['deptid', 'job_code']] = sheet_n[['deptid', 'job_code']].astype('Int64') 

    # remove corporate retained and employee health services
    sheet_clean = sheet_n[~sheet_n['department_description'].str.contains('Corporate Retained|Corp Retained|Employee Health Svcs', case=False, na=False)]  

    # fix duplicate column names  
    cols = pd.Series(sheet_clean.columns)  
    for dup in cols[cols.duplicated()].unique():  
        count = 0  
        for i in range(len(cols)):  
            if cols[i] == dup:  
                count += 1  
                if count > 1:  
                    cols[i] = f"{dup}_{count}"  
    sheet_clean.columns = cols
    return sheet_clean

In [343]:
# clean ppe ftes
ppe_ftes_clean = clean_ftes(ppe_ftes_combined)

# --- budget ---
# select budget related columns
ppe_ftes_budget = ppe_ftes_clean[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'regular_non_prod', 'overtime_2', 'holiday_wkd_2', 'total_2']]

# create fte and type columns 
ppe_ftes_budget['reg_ftes'] = ppe_ftes_budget['total_2'] - ppe_ftes_budget['overtime_2']  
ppe_ftes_budget['ot_ftes'] = ppe_ftes_budget['overtime_2']  
ppe_ftes_budget['total_ftes'] = ppe_ftes_budget['total_2']
ppe_ftes_budget['type'] = 'Budget'

# select columns to keep
ppe_ftes_b = ppe_ftes_budget[['type', 'report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'reg_ftes', 'ot_ftes', 'total_ftes']]

# --- actual ---
ppe_ftes_actual = ppe_ftes_clean[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'regular', 'overtime', 'holiday_wkd', 'non_prod', 'total']]

# remove flex and dialysis
ppe_ftes_rem = ppe_ftes_actual[  
    (~ppe_ftes_actual['job_code_description'].str.contains('FlexStaff', case=False, na=False)) &  
    (~ppe_ftes_actual['department_description'].str.contains('Dialysis', case=False, na=False))  
].reset_index(drop=True)

# create fte and type columns
ppe_ftes_rem['reg_ftes'] = ppe_ftes_rem['total'] - ppe_ftes_rem['overtime']  
ppe_ftes_rem['ot_ftes'] = ppe_ftes_rem['overtime']  
ppe_ftes_rem['total_ftes'] = ppe_ftes_rem['total']
ppe_ftes_rem['type'] = 'Actual'

# select columns to keep
ppe_ftes_a = ppe_ftes_rem[['type', 'report_date', 'bu',  'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'reg_ftes', 'ot_ftes', 'total_ftes']]

# check totals
#display(ppe_ftes_a.groupby(['bu'])[['reg_ftes', 'ot_ftes', 'total_ftes']].sum())

In [344]:
# use the most recent report for ytd
ytd_ftes_recent = ytd_ftes_combined[ytd_ftes_combined['report_date'] == most_recent_report_date]  

# clean ftes
ytd_ftes_clean = clean_ftes(ytd_ftes_recent)

# create fte columns
ytd_ftes_clean['reg_ftes'] = ytd_ftes_clean['total'] - ytd_ftes_clean['overtime']  
ytd_ftes_clean['ot_ftes'] = ytd_ftes_clean['overtime']  
ytd_ftes_clean['total_ftes'] = ytd_ftes_clean['total']
ytd_ftes_clean['type'] = 'Actual YTD'

# select columns to keep
ytd_ftes_a = ytd_ftes_clean[['type', 'report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'reg_ftes', 'ot_ftes', 'total_ftes']]

# 327 gv, 3711 lenox, 416 meeth totals to tie
#display(ytd_ftes_a.groupby(['bu'])[['reg_ftes', 'ot_ftes', 'total_ftes']].sum())

In [345]:
# change ldr column names to fte column names
def rename_ldr(df):
    df = df.rename(columns={  
        'gl_deptid': 'deptid',  
        'description': 'department_description',  
        'jobcode': 'job_code',
        'descr': 'job_code_description'}) 
    return df

# call above function
mtd_ldr_f = rename_ldr(mtd_ldr_flex)
mtd_ldr_d = rename_ldr(mtd_ldr_dial)

# combine all five dataframes
ftes = pd.concat([ytd_ftes_a, ppe_ftes_b, ppe_ftes_a, mtd_ldr_f, mtd_ldr_d], ignore_index=True)

# set deptid and job_code as integers
ftes[['deptid', 'job_code']] = ftes[['deptid', 'job_code']].astype('Int64') 
# set ftes as floats
ftes[['reg_ftes', 'ot_ftes', 'total_ftes']] = ftes[['reg_ftes', 'ot_ftes', 'total_ftes']].fillna(0).astype('float')
# update report date to be end of pay period
ftes['report_date'] = ftes['report_date'].max()

# check totals for ftes
#display(ftes.groupby(['type', 'bu'])[['reg_ftes', 'ot_ftes', 'total_ftes']].sum())

In [346]:
# set conditions for replacement (deptid, job_code, new_dept_id, new_dept_desc)
dept_jc_conditions = [  
    (15600360, 116649, 15601055, 'Access Services'),  
    (15600360, '~116649', 15600180, 'Health Info Mgt'),  
    (29600360, 116649, 29601055, 'Access Services'),  
    (29600360, '~116649', 29600180, 'Health Info Mgt'),
]

# replace dept id and desc where dept id and jc match
for deptid, jc, new_dept_id, new_dept_desc in dept_jc_conditions:  
    if isinstance(jc, str) and jc.startswith('~'):  
        mask = (ftes['deptid'] == deptid) & (ftes['job_code'] != int(jc[1:]))  
    else:  
        mask = (ftes['deptid'] == deptid) & (ftes['job_code'] == jc)  
    ftes.loc[mask, 'deptid'] = new_dept_id  
    ftes.loc[mask, 'department_description'] = new_dept_desc  

# set conditions for replacement (deptid, new_dept_desc)
dept_conditions = [
    (74002564, 'POS - Perianesthesia'),
    (74002545, 'POS - Operating Room'),
    (15600100, 'Hospital Material'),
    (15600160, 'Periop Material')
]

# replace dept desc where dept id matches
for deptid, new_dept_desc in dept_conditions:  
    mask = (ftes['deptid'] == deptid)  
    ftes.loc[mask, 'department_description'] = new_dept_desc  

In [ ]:
# pull xwalk reference file
xwalk = pd.read_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx")

# check for duplicates in crosswalk  
dupes = xwalk[xwalk.duplicated(subset=['dept', 'jc'], keep=False)]
# print dupes if found
if not dupes.empty:  
    print(f"WARNING: {len(dupes)} duplicate dept/jc rows found in crosswalk:")  
    display(dupes.sort_values(['dept', 'jc']))

# create iteration for while loop
iteration = 0  
while True:
    iteration += 1  
    print(f"\n--- Iteration {iteration} ---")

    # deduplicate xwalk before merging  
    xwalk_dedup = (xwalk[['vp', 'director', 'dept', 'dept_desc', 'jc', 'jc_desc']].drop_duplicates(subset=['dept', 'jc'], keep='first'))

    # merge crosswalk with ftes  
    print(f"ftes rows before merge: {len(ftes)}")  
    ftes_lead = pd.merge(xwalk_dedup, ftes, how='right', left_on=['dept', 'jc'], right_on=['deptid', 'job_code'])  
    print(f"ftes_lead rows after merge: {len(ftes_lead)}")

    # check for duplicates in the merge
    if len(ftes_lead) != len(ftes):  
        print("ERROR: Merge duplicated rows. Check crosswalk for duplicate dept/jc combos.")  
        break

    # update job descriptions where crosswalk differs
    jc_update = (ftes_lead['jc_desc'].notna() & ftes_lead['job_code_description'].notna() & (ftes_lead['job_code_description'] != ftes_lead['jc_desc']))  
    ftes_lead.loc[jc_update, 'job_code_description'] = ftes_lead.loc[jc_update, 'jc_desc']  
    print(f"Updated {jc_update.sum()} job descriptions from crosswalk.")

    # split into matched and unmatched  
    jc_nulls = ftes_lead[ftes_lead['jc'].isna()]  
    jc_notnulls = ftes_lead[ftes_lead['jc'].notna()]

    # if all job codes matched
    if jc_nulls.empty:  
        print("All job codes matched. Done.")  
        break

    # after dept lookup, do just jc lookup
    print(f"Found {len(jc_nulls)} unmatched rows. Trying job code only lookup...")

    # try matching nulls by job code alone  
    xwalk_jc_dedup = (xwalk[['vp', 'director', 'jc', 'jc_desc']].drop_duplicates(subset=['jc'], keep='first'))
    jc_repl = pd.merge(xwalk_jc_dedup, jc_nulls[['bu', 'type', 'report_date', 'deptid', 'department_description','job_code', 
                                                 'job_code_description', 'reg_ftes', 'ot_ftes', 'total_ftes']], 
                       how='right', left_on=['jc'], right_on=['job_code'])

    # split into matched and unmatched
    jc_repl_nulls = jc_repl[jc_repl['jc'].isna()]  
    jc_repl_notnulls = jc_repl[jc_repl['jc'].notna()]

    print(f"Job code lookup resolved {len(jc_repl_notnulls)} rows, {len(jc_repl_nulls)} still unmatched.")

    # if there are no nulls
    if jc_repl_nulls.empty:  
        print("All remaining nulls resolved via job code lookup. Done.")
        # combine filled nulls with jc filled nulls  
        ftes_lead = pd.concat([jc_notnulls, jc_repl_notnulls], ignore_index=True)  
        break

    # build new crosswalk rows from unmatched
    dept_leaders = xwalk[['dept', 'dept_desc', 'vp', 'director']].drop_duplicates(subset=['dept'], keep='first')  
    null_rows = jc_repl_nulls[['bu', 'type', 'report_date', 'deptid', 'department_description',  
                                'job_code', 'job_code_description', 'reg_ftes', 'ot_ftes', 'total_ftes']]

    # create new rows for xwalk
    new_rows = pd.merge(dept_leaders, null_rows, how='right', left_on=['dept'], right_on=['deptid'])  
    new_rows = new_rows.rename(columns={'job_code': 'jc', 'job_code_description': 'jc_desc'})

    # keep only crosswalk columns  
    new_rows = new_rows[xwalk.columns].drop_duplicates()

    # append to crosswalk  
    xwalk = pd.concat([xwalk, new_rows], ignore_index=True).drop_duplicates()  
    # write back to crosswalk file
    xwalk.to_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx", index=False)  
    print(f"Added {len(new_rows)} new rows to crosswalk. Re-running...")

    # safety valve  
    if iteration > 10:  
        print("WARNING: Too many iterations. Breaking to avoid infinite loop. Talk to Kate.")  
        break

# final output from while loop
ftes_total = ftes_lead[['vp', 'director', 'bu', 'type', 'report_date', 'deptid',  
                         'department_description', 'job_code', 'job_code_description',  
                         'reg_ftes', 'ot_ftes', 'total_ftes']]

print(f"\nFinal ftes_total: {len(ftes_total)} rows")  
print(f"Total FTEs check: {ftes_total['total_ftes'].sum():.2f} (should match ftes: {ftes['total_ftes'].sum():.2f})") 

# check totals
#display(ftes_total.groupby(['type', 'bu'])[['reg_ftes', 'ot_ftes', 'total_ftes']].sum())

,vp,director,bu,dept,dept_desc,jc,jc_desc
1691,"Vieira, Sara","Wong, Andrew",GVCCC,74002507,Clinical Ancillary,203423,Social Worker - FlexStaff
1692,"Vieira, Sara","Wong, Andrew",GVCCC,74002507,Clinical Ancillary,203423,Social Worker II - FlexStaff



--- Iteration 1 ---
ftes rows before merge: 5237
ftes_lead rows after merge: 5237
Updated 77 job descriptions from crosswalk.
All job codes matched. Done.

Final ftes_total: 5237 rows
Total FTEs check: 13476.45 (should match ftes: 13476.45)


In [333]:
for position control, paste in most recent, add formulas remove zeros in total column

    

SyntaxError: invalid syntax (2224384119.py, line 1)